# Human-in-the-Loop

Até agora, nossos agentes executam ações de forma totalmente autônoma. Isso funciona bem para consultas e buscas, mas em situações críticas — como cancelar uma conta, processar um reembolso ou enviar dinheiro — queremos que um humano aprove a ação antes dela ser executada.

O padrão **Human-in-the-Loop (HITL)** permite que o agente pause a execução em pontos críticos, apresente a ação ao usuário e aguarde aprovação para continuar. Se o usuário rejeitar, o agente segue um caminho alternativo.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Ferramentas com e sem aprovação

Vamos criar um cenário de atendimento ao cliente com tools simples (consultar saldo e fatura) e uma tool crítica (cancelar conta). A tool crítica usa `interrupt()` do LangGraph para pausar e aguardar aprovação.

In [2]:
from langchain.tools import tool
from langgraph.types import interrupt

@tool
def consultar_saldo() -> str:
    """Consulta o saldo atual da conta do cliente."""
    return "Saldo atual: R$ 5.230,00"

@tool
def consultar_fatura() -> str:
    """Consulta a fatura atual do cartao de credito."""
    return "Fatura atual: R$ 1.850,00 (vencimento 15/04/2026)"    

In [3]:
@tool
def cancelar_conta(motivo: str) -> str:
    """Cancela a conta do cliente. Esta acao e irreversivel."""
    aprovacao = interrupt(
        f"APROVACAO NECESSARIA: O cliente deseja cancelar a conta. Motivo: '{motivo}'. Aprovar? (sim/nao)"
    )

    if aprovacao == "sim":
        return f"Conta cancelada com sucesso. Motivo registrado: {motivo}"
    return "Cancelamento nao aprovado. A conta permanece ativa."

A função `interrupt()` faz duas coisas:
1. **Pausa** a execução do agente e salva o estado inteiro no checkpointer
2. **Retorna** uma mensagem que será exibida ao supervisor/usuário

Quando o supervisor responde, a execução continua **exatamente de onde parou** e o valor da resposta é retornado pelo `interrupt()`.

## Criando o agente

O agente precisa de um **checkpointer** para salvar o estado quando for interrompido. Sem ele, não seria possível retomar a execução.

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agente = create_agent(
    model="gpt-4.1-nano",
    tools=[consultar_saldo, consultar_fatura, cancelar_conta],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Voce e um atendente da NovaTech Financeira. "
        "Ajude o cliente com consultas e operacoes na conta. "
        "Para operacoes irreversiveis como cancelamento, use a ferramenta apropriada."
    )
)

## Ação simples (sem interrupção)

Consultas de saldo e fatura são operações seguras. O agente executa normalmente, sem pedir aprovação.

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "atendimento-1"}}

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Qual meu saldo e minha fatura atual?")]},
    config
)

print(resposta["messages"][-1].content)

Seu saldo atual é de R$ 5.230,00 e a fatura atual do seu cartão de crédito é de R$ 1.850,00, com vencimento em 15/04/2026. Se precisar de mais alguma informação, estou à disposição!


## Ação crítica (com interrupção)

Agora vamos pedir o cancelamento da conta. O agente vai chamar a tool `cancelar_conta`, que internamente executa `interrupt()`. A execução para e o agente aguarda aprovação.

In [6]:
config2 = {"configurable": {"thread_id": "atendimento-2"}}

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Quero cancelar minha conta. Estou insatisfeito com o atendimento.")]},
    config2
)

print(resposta["messages"][-1].content)

O agente pausou — a resposta veio vazia porque a execução foi interrompida antes de gerar uma mensagem final. Nenhuma conta foi cancelada ainda. Podemos verificar o estado da interrupção.

In [7]:
state = agente.get_state(config2)

print(f"Proximo passo: {state.next}")
print(f"Interrupcoes pendentes:")
for task in state.tasks:
    if task.interrupts:
        for interrupt_data in task.interrupts:
            print(f"  {interrupt_data.value}")

Proximo passo: ('tools',)
Interrupcoes pendentes:
  APROVACAO NECESSARIA: O cliente deseja cancelar a conta. Motivo: 'insatisfacao com atendimento'. Aprovar? (sim/nao)


O próximo passo é `tools`, confirmando que o agente está pausado dentro da execução da ferramenta. A mensagem da interrupção mostra o motivo do cancelamento e pede aprovação.

Para retomar a execução, usamos `Command(resume=valor)`. O valor passado em `resume` é o que a função `interrupt()` retorna dentro da tool.

In [8]:
from langgraph.types import Command

resposta = agente.invoke(
    Command(resume="sim"),
    config2
)

print(resposta["messages"][-1].content)

Sua conta foi cancelada com sucesso. Se precisar de mais alguma coisa, estou à disposição.


## Rejeitando a ação

Vamos repetir o cenário, mas desta vez rejeitando o cancelamento.

In [9]:
config3 = {"configurable": {"thread_id": "atendimento-3"}}

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Cancela minha conta, nao gostei do app.")]},
    config3
)

print("Agente pausado. Rejeitando...")

Agente pausado. Rejeitando...


In [10]:
resposta = agente.invoke(
    Command(resume="nao"),
    config3
)

print(resposta["messages"][-1].content)

Sua conta não foi cancelada. Posso ajudar com mais alguma coisa?


## Múltiplas tools com interrupt

Podemos ter várias tools críticas no mesmo agente. Cada uma pausa independentemente e aguarda aprovação. Vamos adicionar uma tool de reembolso.

In [11]:
@tool
def solicitar_reembolso(valor: float, descricao: str) -> str:
    """Solicita reembolso de uma compra no cartao de credito."""
    aprovacao = interrupt(
        f"APROVACAO NECESSARIA: Reembolso de R$ {valor:.2f} - {descricao}. Aprovar? (sim/nao)"
    )

    if aprovacao == "sim":
        return f"Reembolso de R$ {valor:.2f} aprovado. Sera creditado na proxima fatura."
    return "Reembolso nao aprovado."


agente2 = create_agent(
    model="gpt-4.1-nano",
    tools=[consultar_saldo, consultar_fatura, cancelar_conta, solicitar_reembolso],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Voce e um atendente da NovaTech Financeira. "
        "Ajude o cliente com consultas e operacoes na conta. "
        "Para operacoes criticas (cancelamento, reembolso), use as ferramentas apropriadas."
    )
)

In [12]:
config4 = {"configurable": {"thread_id": "atendimento-4"}}

resposta = agente2.invoke(
    {"messages": [HumanMessage(content="Fui cobrado R$ 299.90 indevidamente por uma assinatura que ja cancelei. Quero reembolso.")]},
    config4
)

print("Agente pausado. Verificando interrupcao...")

state = agente2.get_state(config4)
for task in state.tasks:
    if task.interrupts:
        for interrupt_data in task.interrupts:
            print(f"  {interrupt_data.value}")

Agente pausado. Verificando interrupcao...
  APROVACAO NECESSARIA: Reembolso de R$ 299.90 - Cobrança indevida por assinatura cancelada. Aprovar? (sim/nao)


In [13]:
resposta = agente2.invoke(
    Command(resume="sim"),
    config4
)

print(resposta["messages"][-1].content)

O reembolso de R$ 299,90 foi aprovado e será creditado na próxima fatura. Se precisar de mais alguma ajuda, estou à disposição.


O padrão HITL é essencial para agentes em produção. Ele permite que o agente tenha autonomia para tarefas simples (consultas, buscas) enquanto mantém supervisão humana para ações críticas e irreversíveis.

O fluxo é sempre o mesmo:
1. O agente chama uma tool que contém `interrupt()`
2. A execução pausa e o estado é salvo no checkpointer
3. O supervisor analisa e responde com `Command(resume=valor)`
4. A execução continua de onde parou, com o valor da aprovação

No próximo notebook, vamos combinar HITL com RAG para criar um sistema completo de atendimento.